# SinglePress (.1pz) — Interactive Demo
This notebook reproduces key results from "SinglePress: A Purpose-Built File Format for Single-Cell Omics Matrices" (DeBruine 2026).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singlet-Bio/singlepress/blob/main/notebooks/demo.ipynb)

In [ ]:
!pip install -q singlepress scipy anndata h5py

In [ ]:
import singlepress
print(f"singlepress v{singlepress.__version__}")

## Download Example Dataset

We use the 10x Genomics PBMC 3k dataset — a standard benchmark in single-cell genomics
(~2,700 PBMCs, ~32,000 genes).

In [ ]:
import urllib.request, os, tempfile, tarfile

url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
tmpdir = tempfile.mkdtemp()
tarpath = os.path.join(tmpdir, "pbmc3k.tar.gz")
urllib.request.urlretrieve(url, tarpath)

with tarfile.open(tarpath) as t:
    t.extractall(tmpdir, filter="data")

mtx_dir = os.path.join(tmpdir, "filtered_gene_bc_matrices", "hg19")
print(f"Downloaded to {mtx_dir}")
print(os.listdir(mtx_dir))

In [ ]:
import scipy.io, scipy.sparse

# Load the Market Matrix file as scipy sparse
mat = scipy.io.mmread(os.path.join(mtx_dir, "matrix.mtx")).T.tocsc()
print(f"Matrix: {mat.shape[0]} cells \u00d7 {mat.shape[1]} genes, {mat.nnz:,} nonzeros")

# Write as 1pz
pz_path = os.path.join(tmpdir, "pbmc3k.1pz")
singlepress.write_1pz(pz_path, mat)

raw_bytes = mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes
pz_bytes = os.path.getsize(pz_path)
print(f"\n1pz file size:    {pz_bytes:>12,} bytes")
print(f"Raw int32 CSC:    {raw_bytes:>12,} bytes")
print(f"Compression ratio: {raw_bytes / pz_bytes:.1f}\u00d7")

In [ ]:
import numpy as np

# Read back and verify bit-exact round-trip
mat2 = singlepress.read_1pz(pz_path)
assert np.array_equal(mat.toarray(), mat2.toarray()), "Round-trip failed!"
print("\u2713 Round-trip verification: PASSED (bit-exact)")

In [ ]:
import singlepress.interop

# Convert directly to AnnData
adata = singlepress.interop.to_anndata(pz_path)
print(f"AnnData: {adata.shape}")
print(f"X type:  {type(adata.X).__name__}")

In [ ]:
import time

# Benchmark read speed (median of 10 trials)
times = []
for _ in range(10):
    t0 = time.perf_counter()
    _ = singlepress.read_1pz(pz_path)
    times.append(time.perf_counter() - t0)

median_ms = sorted(times)[5] * 1000
print(f"Read time (median of 10): {median_ms:.1f} ms")

## Key Results

- **SinglePress achieves ~13\u00d7 compression** on this dataset vs raw int32 CSC arrays
- **Round-trip is bit-exact** — lossless integer-preserving compression
- **Native AnnData integration** via `singlepress.interop.to_anndata()`
- **Sub-millisecond reads** for small datasets; scales linearly with nnz